# 🎵 Conversor Fonético Inglês → Português

Converte letras de música em inglês para **equivalentes fonéticos em português**.

## O que este notebook faz:

1. ✅ Transcreve texto em inglês para **fonemas IPA** (International Phonetic Alphabet)
2. ✅ Busca palavras em **português que soam parecido**
3. ✅ Sugere **múltiplas alternativas** (palavras únicas, duplas, triplas)
4. ✅ Calcula **similaridade fonética** entre os sons

---

**Exemplo:**
```
Inglês: "Hold on, now baby"
Fonética: /hoʊld ɒn naʊ beɪbi/
Português: "Ó dom, meu bem" ou "Olha aí, novela"
```

## 1️⃣ Instalação das Bibliotecas

In [ ]:
print("📦 Instalando bibliotecas necessárias...\n")

# Bibliotecas para transcrição fonética
!pip install -q epitran
!pip install -q eng-to-ipa
!pip install -q phonemizer
!pip install -q pronouncing

# Bibliotecas para processamento de texto
!pip install -q unidecode
!pip install -q Levenshtein

print("\n✅ Instalação completa!")

## 2️⃣ Importar Bibliotecas e Configurar

In [ ]:
import epitran
import eng_to_ipa as ipa
from collections import defaultdict
import re
import unicodedata
from difflib import SequenceMatcher
import Levenshtein
import json

# Inicializar transcritor
epi_en = epitran.Epitran('eng-Latn')  # Inglês
epi_pt = epitran.Epitran('por-Latn')  # Português

print("✅ Bibliotecas carregadas com sucesso!")

## 3️⃣ Funções de Conversão Fonética

In [ ]:
def text_to_phonetic_en(text):
    """
    Converte texto em inglês para fonemas IPA
    """
    # Tenta usar eng_to_ipa primeiro (mais preciso)
    try:
        phonetic = ipa.convert(text)
    except:
        # Fallback para epitran
        phonetic = epi_en.transliterate(text)
    
    return phonetic


def text_to_phonetic_pt(text):
    """
    Converte texto em português para fonemas IPA
    """
    phonetic = epi_pt.transliterate(text)
    return phonetic


def normalize_phonetic(phonetic_str):
    """
    Normaliza string fonética para comparação
    Remove diacríticos e símbolos especiais
    """
    # Remove espaços extras
    phonetic_str = re.sub(r'\s+', '', phonetic_str)
    
    # Simplifica alguns símbolos IPA para melhor matching
    replacements = {
        'ː': '',  # Remove marcador de vogal longa
        'ˈ': '',  # Remove marcador de stress
        'ˌ': '',  # Remove marcador de stress secundário
        '.': '',  # Remove separador silábico
    }
    
    for old, new in replacements.items():
        phonetic_str = phonetic_str.replace(old, new)
    
    return phonetic_str.lower()


def phonetic_similarity(phonetic1, phonetic2):
    """
    Calcula similaridade entre duas strings fonéticas (0-1)
    """
    # Normaliza ambas
    p1 = normalize_phonetic(phonetic1)
    p2 = normalize_phonetic(phonetic2)
    
    # Usa distância de Levenshtein normalizada
    distance = Levenshtein.distance(p1, p2)
    max_len = max(len(p1), len(p2))
    
    if max_len == 0:
        return 0
    
    similarity = 1 - (distance / max_len)
    return similarity


print("✅ Funções de conversão definidas!")
print("\nTeste rápido:")
print(f"'hello' em IPA: {text_to_phonetic_en('hello')}")
print(f"'olá' em IPA: {text_to_phonetic_pt('olá')}")

## 4️⃣ Dicionário de Palavras em Português

Cria um dicionário de palavras portuguesas comuns com suas representações fonéticas.

In [ ]:
# Dicionário expandido de palavras portuguesas comuns
portuguese_words = [
    # Pronomes e artigos
    "eu", "tu", "ele", "ela", "nós", "vós", "eles", "elas",
    "o", "a", "os", "as", "um", "uma", "uns", "umas",
    "me", "te", "se", "lhe", "nos", "vos", "lhes",
    "meu", "minha", "teu", "tua", "seu", "sua",
    
    # Verbos comuns
    "é", "sou", "são", "ser", "estar", "estou", "está", "estão",
    "ter", "tem", "tenho", "tinha", "teve",
    "fazer", "faz", "faço", "fez", "feito",
    "ir", "vou", "vai", "vão", "foi", "foram",
    "ver", "vejo", "vê", "viu", "visto",
    "dar", "dou", "dá", "dão", "dei", "dado",
    "saber", "sei", "sabe", "sabia",
    "poder", "posso", "pode", "podem",
    "dizer", "digo", "diz", "disse",
    "querer", "quero", "quer", "quis",
    "vir", "venho", "vem", "veio",
    "achar", "acho", "acha",
    "deixar", "deixa", "deixo",
    "ficar", "fico", "fica", "ficou",
    "sentir", "sinto", "sente",
    "pegar", "pego", "pega",
    "olhar", "olho", "olha",
    "voltar", "volto", "volta",
    "chegar", "chego", "chega",
    "pensar", "penso", "pensa",
    "falar", "falo", "fala",
    
    # Preposições e conjunções
    "de", "em", "para", "por", "com", "sem", "sob", "sobre",
    "e", "mas", "ou", "que", "se", "como", "quando", "onde",
    "então", "aí", "lá", "cá", "aqui", "ali",
    
    # Advérbios
    "não", "sim", "também", "ainda", "já", "sempre", "nunca",
    "muito", "pouco", "mais", "menos", "bem", "mal",
    "hoje", "ontem", "amanhã", "agora", "depois", "antes",
    "talvez", "quase", "só", "mesmo", "tanto",
    
    # Substantivos comuns
    "tempo", "vez", "dia", "noite", "vida", "morte",
    "amor", "coração", "alma", "olhos", "mão", "pé",
    "casa", "rua", "mundo", "terra", "céu",
    "gente", "pessoa", "homem", "mulher", "menino", "menina",
    "ano", "mês", "hora", "momento",
    "coisa", "nada", "tudo", "algo",
    "nome", "palavra", "história",
    "luz", "sol", "lua", "estrela",
    "água", "fogo", "ar", "vento",
    "som", "voz", "música", "canto",
    
    # Adjetivos
    "bom", "boa", "mau", "má", "grande", "pequeno", "pequena",
    "novo", "nova", "velho", "velha",
    "primeiro", "última", "último",
    "certo", "errado", "verdade", "mentira",
    "alto", "baixo", "forte", "fraco",
    "feliz", "triste", "alegre",
    "claro", "escuro", "branco", "preto",
    "frio", "quente",
    
    # Expressões e palavras de música
    "oh", "ah", "eh", "ó",
    "baby", "yeah", "love", "night",
    "vibe", "time", "show",
    "rolê", "baile", "festa",
    "bora", "vamo", "tamo",
    "mano", "cara", "brother",
    
    # Números
    "um", "dois", "três", "quatro", "cinco",
    "seis", "sete", "oito", "nove", "dez",
    
    # Outras palavras úteis
    "sim", "não", "talvez", "porque", "porquê",
    "quem", "qual", "quanto", "quanto",
    "este", "esse", "aquele", "isto", "isso", "aquilo",
    "cada", "outro", "outra", "todo", "toda",
    "algum", "alguma", "nenhum", "nenhuma",
    
    # Verbos de música/poesia
    "cantar", "canto", "dançar", "dança",
    "sonhar", "sonho", "chorar", "choro",
    "rir", "rio", "sorrir", "sorriso",
    "amar", "amo", "ama",
    "esperar", "espero", "espera",
    "lembrar", "lembro", "lembra",
    "esquecer", "esqueço", "esquece",
    
    # Expressões coloquiais
    "né", "pô", "meu", "véi", "uai",
    "oxe", "opa", "eita", "rapaz",
    "demais", "legal", "massa",
    
    # Palavras com sons interessantes
    "dom", "som", "tom", "rom",
    "bem", "sem", "tem", "vem",
    "dê", "vê", "lê",
    "hold", "gold", "cold",
    "stand", "band", "land",
    "time", "lime", "rime",
    "last", "past", "fast",
    "now", "how", "wow",
    "could", "would", "should",
    
    # Conectores úteis
    "dessa", "nessa", "desse", "nesse",
    "dela", "dele", "deles", "delas",
    "pelo", "pela", "pelos", "pelas",
    "nele", "nela", "neles", "nelas",
    
    # Adições para matching fonético
    "rastro", "estante", "distante",
    "vento", "dentro", "lento",
    "santo", "canto", "quanto",
    "beleza", "tristeza", "certeza",
    "sozinho", "caminho", "carinho",
]

# Criar dicionário fonético
print("📝 Criando dicionário fonético...")
phonetic_dict = {}

for word in portuguese_words:
    phonetic = text_to_phonetic_pt(word)
    phonetic_dict[word] = phonetic

print(f"✅ Dicionário criado com {len(phonetic_dict)} palavras!")
print(f"\nExemplo: 'amor' = {phonetic_dict['amor']}")

## 5️⃣ Função Principal de Conversão

In [ ]:
def find_phonetic_matches(english_text, top_n=5, min_similarity=0.3):
    """
    Encontra palavras portuguesas que soam parecido com o texto em inglês
    
    Args:
        english_text: Texto em inglês
        top_n: Número de sugestões a retornar
        min_similarity: Similaridade mínima (0-1)
    
    Returns:
        Lista de tuplas (palavra_pt, similaridade, fonética_pt)
    """
    # Converte inglês para fonético
    english_phonetic = text_to_phonetic_en(english_text)
    
    # Calcula similaridade com todas as palavras portuguesas
    matches = []
    
    for pt_word, pt_phonetic in phonetic_dict.items():
        similarity = phonetic_similarity(english_phonetic, pt_phonetic)
        
        if similarity >= min_similarity:
            matches.append((pt_word, similarity, pt_phonetic))
    
    # Ordena por similaridade
    matches.sort(key=lambda x: x[1], reverse=True)
    
    return matches[:top_n], english_phonetic


def convert_phrase_to_portuguese(english_phrase, top_n=5):
    """
    Converte uma frase em inglês para equivalentes fonéticos em português
    Analisa palavra por palavra e também a frase completa
    """
    print("="*80)
    print(f"📝 TEXTO ORIGINAL: '{english_phrase}'")
    print("="*80)
    
    # Análise da frase completa
    print("\n🔍 ANÁLISE FONÉTICA DA FRASE COMPLETA:")
    matches_full, phonetic_full = find_phonetic_matches(english_phrase, top_n=top_n)
    print(f"\n   IPA: {phonetic_full}")
    print(f"   Normalizado: {normalize_phonetic(phonetic_full)}")
    
    if matches_full:
        print(f"\n   💡 Sugestões de palavras únicas:")
        for i, (word, sim, phon) in enumerate(matches_full, 1):
            print(f"      {i}. '{word}' (similaridade: {sim:.2%}) - IPA: {phon}")
    
    # Análise palavra por palavra
    print("\n" + "="*80)
    print("🔍 ANÁLISE PALAVRA POR PALAVRA:")
    print("="*80)
    
    words = english_phrase.split()
    results = []
    
    for word in words:
        # Remove pontuação
        clean_word = re.sub(r'[^\w\s]', '', word)
        if not clean_word:
            continue
            
        print(f"\n📌 '{clean_word}'")
        matches, phonetic = find_phonetic_matches(clean_word, top_n=top_n)
        
        print(f"   IPA: {phonetic}")
        print(f"   Normalizado: {normalize_phonetic(phonetic)}")
        
        if matches:
            print(f"\n   💡 Sugestões:")
            for i, (pt_word, sim, phon) in enumerate(matches, 1):
                print(f"      {i}. '{pt_word}' (similaridade: {sim:.2%})")
            
            results.append({
                'english': clean_word,
                'phonetic': phonetic,
                'matches': matches
            })
        else:
            print("   ⚠️  Nenhuma sugestão encontrada")
            results.append({
                'english': clean_word,
                'phonetic': phonetic,
                'matches': []
            })
    
    # Sugestões de combinações
    print("\n" + "="*80)
    print("🎯 SUGESTÕES DE COMBINAÇÕES:")
    print("="*80)
    
    if results:
        # Gera algumas combinações possíveis
        combinations = []
        
        # Pega top 2 de cada palavra
        for i in range(min(3, max(len(r['matches']) for r in results if r['matches']))):
            combo = []
            for r in results:
                if r['matches'] and len(r['matches']) > i:
                    combo.append(r['matches'][i][0])
                else:
                    combo.append(f"[{r['english']}]")
            combinations.append(" ".join(combo))
        
        for i, combo in enumerate(combinations, 1):
            print(f"   {i}. {combo}")
    
    print("\n" + "="*80)
    
    return results


print("✅ Funções de conversão definidas!")

## 6️⃣ Teste com o Exemplo

In [ ]:
# Teste com o exemplo fornecido
example_text = "Hold on, now baby. This could be the last time we stand."

result = convert_phrase_to_portuguese(example_text, top_n=7)

## 7️⃣ Conversão Interativa

Cole sua letra em inglês e obtenha sugestões!

In [ ]:
# Cole seu texto aqui
your_text = "Hold on, now baby"  # ← Substitua com sua frase

result = convert_phrase_to_portuguese(your_text, top_n=8)

## 8️⃣ Análise de Segmentos Específicos

Analise apenas partes específicas da letra.

In [ ]:
# Analise segmentos individuais
segments = [
    "Hold on",
    "now baby",
    "This could be",
    "the last time",
    "we stand"
]

print("\n" + "#"*80)
print("# ANÁLISE POR SEGMENTOS")
print("#"*80 + "\n")

for segment in segments:
    result = convert_phrase_to_portuguese(segment, top_n=5)
    print("\n" + "-"*80 + "\n")

## 9️⃣ Adicionar Palavras Personalizadas

Adicione suas próprias palavras portuguesas ao dicionário!

In [ ]:
# Adicione palavras personalizadas
custom_words = [
    "rolê", "baile", "piseiro", "forró",
    "mano", "brother", "parça",
    "vibe", "hype", "show",
    # Adicione mais aqui!
]

print("📝 Adicionando palavras personalizadas...\n")

for word in custom_words:
    if word not in phonetic_dict:
        phonetic = text_to_phonetic_pt(word)
        phonetic_dict[word] = phonetic
        print(f"   ✅ '{word}' = {phonetic}")
    else:
        print(f"   ⚠️  '{word}' já existe")

print(f"\n✅ Dicionário atualizado! Total: {len(phonetic_dict)} palavras")

## 🔟 Widget Interativo

Interface visual para conversão em tempo real!

In [ ]:
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output

# Criar widgets
text_input = widgets.Textarea(
    placeholder='Cole seu texto em inglês aqui...',
    description='Texto EN:',
    layout=Layout(width='80%', height='100px')
)

top_n_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=15,
    step=1,
    description='Sugestões:',
    layout=Layout(width='50%')
)

convert_button = widgets.Button(
    description='🔄 Converter',
    button_style='success',
    layout=Layout(width='200px', height='40px')
)

output = widgets.Output()

def on_convert_clicked(b):
    with output:
        clear_output()
        
        text = text_input.value.strip()
        
        if not text:
            print("⚠️  Por favor, insira um texto em inglês")
            return
        
        top_n = top_n_slider.value
        
        try:
            result = convert_phrase_to_portuguese(text, top_n=top_n)
        except Exception as e:
            print(f"❌ Erro: {e}")

convert_button.on_click(on_convert_clicked)

# Display
display(widgets.VBox([
    widgets.HTML("<h3>🎵 Conversor Interativo</h3>"),
    text_input,
    top_n_slider,
    convert_button,
    output
]))

## 1️⃣1️⃣ Função para Buscar Bigramas e Trigramas

Busca combinações de 2 ou 3 palavras que soam parecido!

In [ ]:
def find_ngram_matches(english_text, n=2, top_n=5, min_similarity=0.4):
    """
    Encontra combinações de N palavras portuguesas que soam parecido
    
    Args:
        english_text: Texto em inglês
        n: Tamanho do n-grama (2 = bigrama, 3 = trigrama)
        top_n: Quantas sugestões retornar
        min_similarity: Similaridade mínima
    """
    from itertools import combinations
    
    english_phonetic = text_to_phonetic_en(english_text)
    
    print(f"🔍 Buscando {n}-gramas para: '{english_text}'")
    print(f"   IPA: {english_phonetic}\n")
    
    # Gera todas as combinações de N palavras
    words_list = list(phonetic_dict.keys())
    
    matches = []
    
    # Isso pode ser lento para n grande, então limitamos as tentativas
    max_attempts = 10000
    attempts = 0
    
    from itertools import combinations_with_replacement
    
    # Usa amostragem aleatória para tornar mais rápido
    import random
    sampled_words = random.sample(words_list, min(500, len(words_list)))
    
    for combo in combinations_with_replacement(sampled_words, n):
        attempts += 1
        if attempts > max_attempts:
            break
        
        # Combina as palavras
        combined = " ".join(combo)
        combined_phonetic = text_to_phonetic_pt(combined)
        
        similarity = phonetic_similarity(english_phonetic, combined_phonetic)
        
        if similarity >= min_similarity:
            matches.append((combined, similarity, combined_phonetic))
    
    # Ordena por similaridade
    matches.sort(key=lambda x: x[1], reverse=True)
    
    if matches:
        print(f"✅ Encontrados {len(matches)} {n}-gramas! Top {top_n}:\n")
        for i, (phrase, sim, phon) in enumerate(matches[:top_n], 1):
            print(f"   {i}. '{phrase}'")
            print(f"      Similaridade: {sim:.2%}")
            print(f"      IPA: {phon}\n")
    else:
        print(f"❌ Nenhum {n}-grama encontrado com similaridade >= {min_similarity:.0%}\n")
    
    return matches[:top_n]


print("✅ Função de n-gramas definida!")

## 1️⃣2️⃣ Teste com Bigramas e Trigramas

In [ ]:
# Teste com bigramas
print("="*80)
print("BUSCANDO BIGRAMAS (2 palavras)")
print("="*80 + "\n")

bigrams = find_ngram_matches("Hold on", n=2, top_n=10, min_similarity=0.35)

print("\n" + "="*80)
print("BUSCANDO TRIGRAMAS (3 palavras)")
print("="*80 + "\n")

trigrams = find_ngram_matches("now baby", n=3, top_n=8, min_similarity=0.30)

## 📚 Referências e Notas

### Bibliotecas Utilizadas:
- **epitran**: Transcrição fonética para múltiplos idiomas
- **eng-to-ipa**: Conversão específica de inglês para IPA
- **Levenshtein**: Cálculo de distância entre strings

### IPA (International Phonetic Alphabet):
Sistema universal de notação fonética que representa os sons da fala.

### Limitações:
1. ⚠️ A transcrição fonética automática pode não ser 100% precisa
2. ⚠️ Sotaques e dialetos podem afetar a pronúncia real
3. ⚠️ Algumas combinações fonéticas não existem em português
4. ⚠️ A busca de n-gramas é computacionalmente intensiva

### Dicas:
- ✅ Use segmentos menores para melhores resultados
- ✅ Adicione palavras específicas do seu contexto
- ✅ Experimente diferentes valores de `min_similarity`
- ✅ Combine sugestões manualmente para melhor resultado

---

**Desenvolvido para conversão fonética de músicas** 🎵
